# Exemplo: Classificação multilabel
--------------------------------

Este exemplo mostra como usar o ExperionML para resolver um problema de classificação multilabel.

Os dados utilizados são um conjunto sintético criado com a função [make_multilabel_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_multilabel_classification.html) do sklearn.

## Carregar os dados

In [1]:
# Import packages
import pandas as pd
from experionml import ExperionMLClassifier
from sklearn.datasets import make_multilabel_classification

In [2]:
# Create data
X, y = make_multilabel_classification(n_samples=300, n_classes=3, random_state=1)

## Executar o pipeline

In [3]:
# Observe que, para tarefas multioutput, você deve especificar o argumento `y`
experionml = ExperionMLClassifier(X, y=y, verbose=2, random_state=1)

<< ================== ExperionML ================== >>

Configuração ==================== >>
Tarefa do algoritmo: Multilabel classification.

Estatísticas do conjunto de dados ==================== >>
Formato: (300, 23)
Tamanho do conjunto de train: 240
Tamanho do conjunto de test: 60
-------------------------------------
Memória: 55.33 kB
Escalonado: False
Valores atípicos: 35 (0.6%)



In [4]:
# Mostre os modelos que oferecem suporte nativo a tarefas multilabel
experionml.available_models(native_multilabel=True)

,acronym,fullname,estimator,module,handles_missing,needs_scaling,accepts_sparse,native_multilabel,native_multioutput,validation,supports_engines
0,Tree,DecisionTree,DecisionTreeClassifier,sklearn.tree._classes,True,False,True,True,True,None,sklearn
1,ETree,ExtraTree,ExtraTreeClassifier,sklearn.tree._classes,False,False,True,True,True,None,sklearn
2,ET,ExtraTrees,ExtraTreesClassifier,sklearn.ensemble._forest,False,False,True,True,True,None,sklearn
3,KNN,KNearestNeighbors,KNeighborsClassifier,sklearn.neighbors._classification,False,True,True,True,True,None,"sklearn, sklearnex, cuml"
4,MLP,MultiLayerPerceptron,MLPClassifier,sklearn.neural_network._multilayer_perceptron,False,True,True,True,False,max_iter,sklearn
5,RNN,RadiusNearestNeighbors,RadiusNeighborsClassifier,sklearn.neighbors._classification,False,True,True,True,True,None,sklearn
6,RF,RandomForest,RandomForestClassifier,sklearn.ensemble._forest,False,False,True,True,True,None,"sklearn, sklearnex, cuml"
7,Ridge,Ridge,RidgeClassifier,sklearn.linear_model._ridge,False,True,True,True,False,None,"sklearn, sklearnex, cuml"


In [5]:
experionml.run(models=["LDA", "RF"], metric="recall_weighted")


Training ========================= >>
Models: LDA, RF
Metric: recall_weighted


Results for LinearDiscriminantAnalysis:
Fit ---------------------------------------------
Train evaluation --> recall_weighted: 0.8912
Test evaluation --> recall_weighted: 0.899
Time elapsed: 0.052s
-------------------------------------------------
Time: 0.052s


Results for RandomForest:
Fit ---------------------------------------------


Train evaluation --> recall_weighted: 1.0
Test evaluation --> recall_weighted: 0.9091
Time elapsed: 0.174s
-------------------------------------------------
Time: 0.174s


Resultados finais ==================== >>
Tempo total: 0.244s
-------------------------------------
LinearDiscriminantAnalysis --> recall_weighted: 0.899
RandomForest               --> recall_weighted: 0.9091 !


In [6]:
# Observe que modelos multioutput não nativos usam um wrapper meta-estimador
print(f"Estimator for LDA is: {experionml.lda.estimator}")
print(f"Estimator for RF is: {experionml.rf.estimator}")

Estimator for LDA is: ClassifierChain(base_estimator=LinearDiscriminantAnalysis(), random_state=1)
Estimator for RF is: RandomForestClassifier(n_jobs=1, random_state=1)


## Adicionar modelos multilabel personalizados

Para usar seu próprio metaestimador com parâmetros personalizados, adicione-o como um [modelo personalizado](https://tvdboom.github.io/ExperionML/latest/user_guide/models/#custom-models).
Também é possível ajustar os hiperparâmetros desse metaestimador personalizado.

In [7]:
from experionml import ExperionMLModel
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegression
from optuna.distributions import CategoricalDistribution, IntDistribution

custom_model = ExperionMLModel(
    estimator=ClassifierChain(LogisticRegression(), cv=3),
    name="chain",
    needs_scaling=True,
    native_multilabel=True,
)

experionml.run(
    models=custom_model,
    n_trials=5,
    ht_params={
        "distributions": {
            "order": CategoricalDistribution([[0, 1, 2], [2, 1, 0], [1, 2, 0]]),
            "base_estimator__max_iter": IntDistribution(100, 200, step=10),
            "base_estimator__solver": CategoricalDistribution(["lbfgs", "newton-cg"]),            
        }
    }
)


Training ========================= >>
Models: chain
Metric: recall_weighted


Running hyperparameter tuning for ClassifierChain...
| trial |     order | base_estimator__max_iter | base_estimator__solver | recall_weighted | best_recall_weighted | time_trial | time_ht |    state |
| ----- | --------- | ------------------------ | ---------------------- | --------------- | -------------------- | ---------- | ------- | -------- |


| 0     | [2, 1, 0] |                      130 |                  lbfgs |          0.8205 |               0.8205 |     0.168s |  0.168s | COMPLETE |


| 1     | [1, 2, 0] |                      150 |              newton-cg |          0.8205 |               0.8205 |     0.098s |  0.265s | COMPLETE |
| 2     | [2, 1, 0] |                      170 |              newton-cg |          0.8205 |               0.8205 |     0.073s |  0.338s | COMPLETE |


| 3     | [1, 2, 0] |                      200 |              newton-cg |          0.8205 |               0.8205 |     0.075s |  0.413s | COMPLETE |


| 4     | [2, 1, 0] |                      100 |              newton-cg |          0.8205 |               0.8205 |     0.073s |  0.486s | COMPLETE |
Hyperparameter tuning ---------------------------
Best trial --> 0
Best parameters:
 --> order: [2, 1, 0]
 --> base_estimator__max_iter: 130
 --> base_estimator__solver: lbfgs
Best evaluation --> recall_weighted: 0.8205
Time elapsed: 0.486s
Fit ---------------------------------------------


Train evaluation --> recall_weighted: 0.8964
Test evaluation --> recall_weighted: 0.9192
Time elapsed: 0.333s
-------------------------------------------------
Time: 0.819s


Resultados finais ==================== >>
Tempo total: 0.854s
-------------------------------------
ClassifierChain --> recall_weighted: 0.9192


## Analisar os resultados

In [8]:
experionml.rf.evaluate()

accuracy              0.6333
ap                    0.9120
f1_weighted           0.8608
jaccard_weighted      0.7802
precision_weighted    0.8711
recall_weighted       0.9091
auc                   0.9167
Name: RF, dtype: float64

In [9]:
# Use o parâmetro target nos gráficos para especificar qual coluna alvo usar
experionml.plot_roc(target=2)

In [10]:
# Quando o parâmetro target também especifica a classe, use o formato (coluna, classe)
experionml.plot_probabilities(models="chain", target=(2, 1))

In [11]:
with experionml.canvas(figsize=(900, 600)):
    experionml.plot_calibration(target=0)
    experionml.plot_calibration(target=1)